# Training Notebook (Colab / Local)

Config-driven training pipeline. Core logic lives in `data_splitter.py`, `dataset.py`,
`model.py`, `pytorch_lightning.py`, and `training_utils.py` — this notebook just wires
them together against `configs/base.yaml`. Edit those files directly; local runs (no
`google.colab` import) pick up changes immediately, no push/pull needed.

**Kaggle auth** (only needed if `data/` isn't already present): tries, in order, an
existing `KAGGLE_API_TOKEN` env var, `~/.kaggle/access_token`, `~/.kaggle/kaggle.json`,
a Colab secret named `KAGGLE_API_TOKEN` (browser UI only), then an interactive prompt.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'

try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content/beilinson')
    if PROJECT_ROOT.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT)])
except ImportError:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))

Project root: /content/beilinson
Has data already: False


In [2]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

0

In [ ]:
import importlib
import sys

import yaml
import torch
import pandas as pd
import lightning.pytorch as pl
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

# Force-reload our own modules (not third-party ones) in dependency order, so re-running
# this cell after a `git pull` picks up the latest code even in an already-running kernel -
# a plain `import`/`from X import Y` is a no-op once a module is already in sys.modules.
for _module_name in ['data_splitter', 'model', 'pytorch_lightning', 'dataset', 'training_utils']:
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])
    else:
        importlib.import_module(_module_name)

from dataset import MultiClipWorkoutDataset, WorkoutSequenceDataset
from model import SequenceClassifier
from pytorch_lightning import WorkoutLightningModule
from training_utils import (
    classification_metrics,
    ensure_artifacts,
    ensure_dataset,
    ensure_image_cache,
    epoch_history,
    evaluate_multi_clip,
    plot_confusion_matrix,
    save_results,
)

## Config

`CONFIG_PATH` picks which yaml file under `configs/` to run - point it at a different file
(e.g. a copy of `base.yaml` with a different `backbone`) to run a different experiment
without editing `base.yaml` itself.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'base.yaml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

print('Using config:', CONFIG_PATH)
CONFIG

## Dataset

Downloads from Kaggle only if `data/` is empty (see `ensure_dataset` in `training_utils.py`
for the auth fallback chain).

In [5]:
class_names = ensure_dataset(PROJECT_ROOT)
print(f'{len(class_names)} classes:', class_names)

100%|██████████| 818M/818M [00:08<00:00, 103MB/s] 

Extracting files...


22 classes: ['barbell biceps curl', 'bench press', 'chest fly machine', 'deadlift', 'decline bench press', 'hammer curl', 'hip thrust', 'incline bench press', 'lat pulldown', 'lateral raises', 'leg extension', 'leg raises', 'plank', 'pull up', 'push up', 'romanian deadlift', 'russian twist', 'shoulder press', 'squat', 't bar row', 'tricep dips', 'tricep pushdown']


## Build dataloaders

`ensure_artifacts` builds (or reuses) the clip/frame manifests and the fixed-length
`f00..fNN` frame-sequence CSV. `ensure_image_cache` pre-resizes every image to `image_size`
once into `artifacts/image_cache_<size>/` (mirroring `data/`'s layout) so later epochs load
already-resized files instead of re-decoding the original (often much larger) JPEGs every
time - measured ~1.6x faster per image. Each split gets its own `WorkoutSequenceDataset`,
wrapped in a `DataLoader`.

In [ ]:
pl.seed_everything(CONFIG['mode']['seed'], workers=True)

artifacts = ensure_artifacts(CONFIG, PROJECT_ROOT)

data_cfg = CONFIG['data']
image_size = data_cfg['image_size']
batch_size = data_cfg['batch_size']
num_workers = data_cfg['num_workers']
pin_memory = torch.cuda.is_available()

cached_data_root = ensure_image_cache(PROJECT_ROOT, image_size)

train_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], cached_data_root, split='train', image_size=image_size)
val_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], cached_data_root, split='val', image_size=image_size)
test_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], cached_data_root, split='test', image_size=image_size)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)

label_map = pd.read_csv(artifacts['label_map'])
num_classes = int(label_map['label_id'].nunique())

print(f'{num_classes} classes')
print(f'train {len(train_dataset)} / val {len(val_dataset)} / test {len(test_dataset)} clips')

## Build model

`SequenceClassifier` runs a per-frame encoder over every frame independently, then pools
the per-frame embeddings across time (mean/max/LSTM, per `temporal_pooling`) before a
linear classifier head. `WorkoutLightningModule` wraps it with the train/val/test/predict
steps and the optimizer.

The per-frame encoder is controlled by `model.backbone` in the config:
- `custom` — the original small from-scratch CNN (`FrameEncoder`), trained end to end.
- `resnet18` / `mobilenet_v3_small` / `efficientnet_b0` — an ImageNet-pretrained
  torchvision backbone (`PretrainedFrameEncoder`) with a projection head on top.
  `mobilenet_v3_small`/`efficientnet_b0` are the recommended choices for a dataset this
  small (~1100 clips) — fewer params than resnet18, less prone to overfitting, and fast.
  `freeze_backbone: true` (default) trains only the projection/classifier head on top of
  frozen pretrained features; set it `false` to fine-tune the backbone too (slower, more
  data-hungry, only worth trying once the frozen version is working).

In [ ]:
model_cfg = CONFIG['model']
training_cfg = CONFIG['training']

model = SequenceClassifier(
    num_classes=num_classes,
    in_channels=model_cfg['in_channels'],
    hidden_dims=tuple(model_cfg['hidden_dims']),
    embedding_dim=model_cfg['embedding_dim'],
    dropout=model_cfg['dropout'],
    temporal_pooling=model_cfg['temporal_pooling'],
    backbone=model_cfg.get('backbone', 'custom'),
    freeze_backbone=model_cfg.get('freeze_backbone', True),
)
lit_module = WorkoutLightningModule(
    model=model,
    lr=training_cfg['lr'],
    weight_decay=training_cfg['weight_decay'],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'backbone={model_cfg.get("backbone", "custom")}  trainable params: {trainable:,} / {total:,}')
lit_module

## Build trainer

Callbacks: `ModelCheckpoint` keeps the best epoch by `monitor`/`monitor_mode`,
`EarlyStopping` stops after `patience` epochs without improvement, `LearningRateMonitor`
logs the LR each epoch. `precision: auto` picks fp16 on GPU, fp32 on CPU.

Two loggers: `CSVLogger` (what `epoch_history` reads back afterward - listed first so
`trainer.logger` still resolves to it) and `TensorBoardLogger`, so you can watch loss/acc
live in TensorBoard while `Trainer.fit` runs below, instead of only seeing it after the
fact.

In [ ]:
checkpoint_dir = PROJECT_ROOT / training_cfg.get('checkpoint_dir', 'artifacts/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

monitor = training_cfg.get('monitor', 'val_acc')
monitor_mode = training_cfg.get('monitor_mode', 'max')

callbacks = [
    ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename='epoch{epoch:02d}-{val_acc:.3f}',
        monitor=monitor,
        mode=monitor_mode,
        save_top_k=1,
    ),
    EarlyStopping(monitor=monitor, mode=monitor_mode, patience=training_cfg.get('patience', 4)),
    LearningRateMonitor(logging_interval='epoch'),
]

precision = training_cfg.get('precision', '32-true')
if precision == 'auto':
    precision = '16-mixed' if torch.cuda.is_available() else '32-true'

loggers = [
    CSVLogger(save_dir=str(PROJECT_ROOT / 'artifacts')),
    TensorBoardLogger(save_dir=str(PROJECT_ROOT / 'artifacts'), name='tb_logs'),
]

trainer = pl.Trainer(
    max_epochs=training_cfg.get('max_epochs', 10),
    accelerator=training_cfg.get('accelerator', 'auto'),
    devices=training_cfg.get('devices', 'auto'),
    precision=precision,
    log_every_n_steps=training_cfg.get('log_every_n_steps', 10),
    default_root_dir=str(PROJECT_ROOT / 'artifacts'),
    logger=loggers,
    callbacks=callbacks,
)

## Live training dashboard (TensorBoard)

Launch this before `Trainer.fit` below - it updates live as training logs each epoch
(reads from `artifacts/tb_logs`, the `TensorBoardLogger` dir set up above).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PROJECT_ROOT / 'artifacts' / 'tb_logs'}

## Train

Live progress bar with per-step loss/accuracy comes from Lightning's `Trainer.fit`
directly below.

In [9]:
trainer.fit(lit_module, train_loader, val_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SequenceClassifier │  113 K │ train │     0 │
└───┴───────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 113 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 113 K                                                                                                
Total estimated model params size (MB): 0.452                                                                      
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 3. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 6. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


## Training history

Per-epoch train/val loss and accuracy, read back from the CSV logger.

In [10]:
epoch_history(trainer)

FileNotFoundError: [Errno 2] No such file or directory: '/content/beilinson/artifacts/lightning_logs/version_0/metrics.csv'

## Evaluate

In [ ]:
test_results = trainer.test(lit_module, dataloaders=test_loader, verbose=True)

## Predict & save

In [ ]:
prediction_batches = trainer.predict(lit_module, dataloaders=test_loader)
summary = save_results(trainer, artifacts, test_results, prediction_batches, PROJECT_ROOT, cfg=CONFIG)
summary

In [ ]:
import json

summary_path = PROJECT_ROOT / 'artifacts' / 'training_summary.json'
with open(summary_path, 'r', encoding='utf-8') as handle:
    summary = json.load(handle)

summary

## Confusion matrix & per-class metrics (test set)

`predictions.csv` (just written by `save_results`) has one row per test clip with its
true `label` and predicted `prediction`. `classification_metrics` turns that into a
confusion matrix and per-class precision/recall/F1 (via scikit-learn).

In [ ]:
predictions = pd.read_csv(artifacts['sequence_manifest'].parent / 'predictions.csv')
class_names_ordered = label_map.sort_values('label_id')['class'].tolist()

confusion_df, report_df = classification_metrics(predictions, class_names_ordered)
plot_confusion_matrix(confusion_df, title='Test set confusion matrix')
report_df

## Multi-clip evaluation (validation set)

Checks whether averaging predictions over multiple windows per clip (instead of the single
evenly-spaced window every clip gets above) actually helps on *this* dataset, before relying
on it anywhere else. `MultiClipWorkoutDataset` splits each clip into `NUM_CLIPS` contiguous
segments and samples a window from each (see `sample_or_pad_indices_multi` in
`data_splitter.py`); `evaluate_multi_clip` averages the softmax predictions across those
windows per clip. Run on the validation set (not test) so test stays untouched for a final,
one-time check later.

In [ ]:
from dataset import MultiClipWorkoutDataset
from training_utils import epoch_history, evaluate_multi_clip

NUM_CLIPS = 5

multi_clip_val_dataset = MultiClipWorkoutDataset(
    frame_manifest_path=artifacts['frame_manifest'],
    label_map_path=artifacts['label_map'],
    data_root=cached_data_root,
    split='val',
    sequence_len=data_cfg['sequence_len'],
    num_clips=NUM_CLIPS,
    image_size=data_cfg['image_size'],
)
multi_clip_val_accuracy, multi_clip_val_results = evaluate_multi_clip(
    lit_module, multi_clip_val_dataset, batch_size=max(1, batch_size // NUM_CLIPS),
)

single_window_val_acc = float(epoch_history(trainer)['val_acc'].iloc[-1])
print(f'Single-window val_acc (last epoch): {single_window_val_acc:.4f}')
print(f'Multi-clip (num_clips={NUM_CLIPS}) val_acc:        {multi_clip_val_accuracy:.4f}')
print(f'Delta: {multi_clip_val_accuracy - single_window_val_acc:+.4f}')